# 12차시 실습 — 내 MVP를 '팀'으로 (Planner + Executor + 미니 하네스)

> **day1 연속 실습.** 7차시에서 만든 **한 명의 에이전트**(맛집 추천 도우미)를,
> 오늘은 **계획자(Planner) + 실행자(Executor)** 로 나누고, **관리자(Manager)** 가 작업을 라우팅하게 만듭니다.
> 마지막에 **공유 규칙(AGENTS.md) + 검증 게이트(validate)** 라는 **미니 하네스**를 코드로 얹어,
> 여러 에이전트가 **같은 기준에 정렬**되고 **오류가 전파되지 않게** 막아 봅니다.

## 🧪 사용법
- 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**하는 동반 자료입니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다. 위에서부터 실행하세요.
- 7차시의 **도구(function tool)와 ReAct 루프를 그대로 재사용**합니다 — 오늘은 그걸 '팀'으로 키웁니다.

In [58]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오

7차시: **한 에이전트**가 추론→도구→관찰을 혼자 반복(ReAct)했습니다.
12차시: 같은 일을 **팀**으로 — 계획자가 목표를 단계로 쪼개고(핸드오프), 실행자가 도구로 처리하고, 관리자가 종합합니다.

> 공통 예시 = **맛집 추천 도우미**(7차시와 동일). 구조는 그대로이니 **당신 팀 MVP의 도구/규칙**으로 바꾸면 적용됩니다(STEP 6).

## STEP 1 — 도구(tool) 재사용 — *7차시 그대로*

에이전트의 **손발**입니다. 7차시에서 정의한 도구를 **그대로** 가져옵니다 — 팀으로 키워도 도구는 공유 자산입니다.

⌨️ **터미널 Codex:** `codex "7차시 맛집 도구(search_restaurants·budget_check)를 그대로 쓰는 Planner+Executor 팀을 만들어줘"`

In [59]:
# 7차시와 동일한 도구 (실제로는 day1 앱의 DB/API 자리)
RESTAURANTS = {
  "강남": [{"name":"매운갈비찜집","price":18000,"taste":"매움"},
          {"name":"순한국밥","price":9000,"taste":"순함"},
          {"name":"불막창","price":22000,"taste":"매움"}],
  "홍대": [{"name":"치즈파스타","price":15000,"taste":"순함"}],
}
def search_restaurants(area: str, taste: str = "") -> str:
    items = RESTAURANTS.get(area, [])
    if taste: items = [r for r in items if r["taste"]==taste]
    return json.dumps(items, ensure_ascii=False) if items else "결과 없음"
def budget_check(price: int, people: int) -> str:
    return f"1인당 {price//max(people,1):,}원"

TOOL_IMPL = {"search_restaurants": search_restaurants, "budget_check": budget_check}
TOOLS = [
 {"type":"function","function":{"name":"search_restaurants","description":"지역(과 맛)으로 맛집 검색",
   "parameters":{"type":"object","properties":{"area":{"type":"string"},"taste":{"type":"string","description":"매움/순함, 선택"}},"required":["area"]}}},
 {"type":"function","function":{"name":"budget_check","description":"총가격과 인원으로 1인당 가격 계산",
   "parameters":{"type":"object","properties":{"price":{"type":"integer"},"people":{"type":"integer"}},"required":["price","people"]}}},
]
print("도구 준비(7차시 재사용):", list(TOOL_IMPL))

도구 준비(7차시 재사용): ['search_restaurants', 'budget_check']


## STEP 2 — Planner(계획자): 목표 → 단계 분해

큰 그림을 그리는 역할. 사용자 목표를 **독립적인 하위작업**으로 쪼개 JSON으로 내놓습니다.
계획이 **명시적**이라 어디서 실패했는지 추적·재계획이 쉽습니다(building 5 Planner-Executor).

In [60]:
def plan(goal, rules=""):
    sys = ("너는 Planner(계획자)다. 사용자 목표를 실행 가능한 하위작업으로 분해한다. "
           'JSON으로만 답하라: {"steps": ["하위작업1", "하위작업2"]} (3개 이내, 간결).')
    if rules: sys += "\n[하네스 규칙]\n" + rules
    out = client.chat.completions.create(
        model=MODEL,
        messages=[{"role":"system","content":sys},{"role":"user","content":goal}],
        response_format={"type":"json_object"},
    ).choices[0].message.content
    steps = json.loads(out).get("steps", [])
    print("🧭 [PLAN] 목표를", len(steps), "단계로 분해:")
    for i, s in enumerate(steps, 1):
        print(f"   {i}. {s}")
    return steps

GOAL = "강남에서 매운 점심을 2명이서 먹고 싶어. 추천과 1인당 가격까지 알려줘."
_ = plan(GOAL)

🧭 [PLAN] 목표를 3 단계로 분해:
   1. 강남 지역의 매운 음식점 목록 검색
   2. 음식점의 인기 메뉴와 가격 확인
   3. 1인당 가격이 적정한 음식점 선택


## STEP 3 — Executor(실행자) + 핸드오프

손발 역할. **7차시 ReAct 루프를 그대로 재사용**해, 넘겨받은 하위작업 하나를 도구로 처리하고 짧게 보고합니다.
계획자→실행자로 일이 넘어가는 순간이 **핸드오프**입니다.

In [61]:
def executor(subtask, rules="", max_steps=4, verbose=True):
    sys = "너는 Executor(실행자)다. 넘겨받은 하위작업을 도구로 처리하고 한국어로 짧게 결과만 보고하라."
    if rules: sys += "\n[하네스 규칙]\n" + rules
    messages = [{"role":"system","content":sys},{"role":"user","content":subtask}]
    for step in range(1, max_steps+1):
        m = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS).choices[0].message
        messages.append(m)
        if not m.tool_calls:
            return m.content
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments); result = TOOL_IMPL[tc.function.name](**args)
            if verbose: print(f"   🔧 {tc.function.name}({args}) → 👀 {result}")
            messages.append({"role":"tool","tool_call_id":tc.id,"content":str(result)})
    return "최대 스텝 도달"

# 단독 테스트: 실행자에게 하위작업 하나를 핸드오프
print("🤝 [HANDOFF] planner → executor")
print("   ✅ 보고:", executor("강남에서 매운 맛집을 검색해줘"))

🤝 [HANDOFF] planner → executor
   🔧 search_restaurants({'area': '강남', 'taste': '매움'}) → 👀 [{"name": "매운갈비찜집", "price": 18000, "taste": "매움"}, {"name": "불막창", "price": 22000, "taste": "매움"}]
   ✅ 보고: 강남의 매운 맛집은 다음과 같습니다:
1. 매운갈비찜집 - 18,000원
2. 불막창 - 22,000원


## STEP 4 — Manager(관리자) 라우팅 루프

관리자-작업자 패턴. 관리자가 **계획의 각 단계를 실행자에게 라우팅**하고, 보고를 모아 **종합**합니다.
(여기선 작업자가 하나지만, 같은 구조로 검색·계산·리뷰 전문가로 늘릴 수 있습니다.)

In [62]:
def manager(goal, rules="", verbose=True):
    steps = plan(goal, rules)
    observations = []
    for i, task in enumerate(steps, 1):
        if verbose: print(f"\n🤝 [HANDOFF] manager → executor (작업 {i}/{len(steps)}): {task}")
        obs = executor(task, rules, verbose=verbose)
        if verbose: print(f"   ✅ 보고: {obs}")
        observations.append(f"- {task} → {obs}")
    sys = "너는 Manager(관리자)다. 작업자들의 보고를 종합해 사용자에게 최종 답을 한국어로 제시하라."
    if rules: sys += "\n[하네스 규칙]\n" + rules
    final = client.chat.completions.create(
        model=MODEL,
        messages=[{"role":"system","content":sys},
                  {"role":"user","content":f"목표: {goal}\n작업 보고:\n" + "\n".join(observations)}],
    ).choices[0].message.content
    return final

print("\n📋 [최종 종합]\n", manager(GOAL))

🧭 [PLAN] 목표를 3 단계로 분해:
   1. 강남 지역의 매운 음식점 검색
   2. 추천 음식점 목록 작성
   3. 1인당 가격 조사

🤝 [HANDOFF] manager → executor (작업 1/3): 강남 지역의 매운 음식점 검색
   🔧 search_restaurants({'area': '강남', 'taste': '매움'}) → 👀 [{"name": "매운갈비찜집", "price": 18000, "taste": "매움"}, {"name": "불막창", "price": 22000, "taste": "매움"}]
   ✅ 보고: 강남의 매운 음식점은 다음과 같습니다:
1. 매운갈비찜집 - 가격: 18,000원
2. 불막창 - 가격: 22,000원

🤝 [HANDOFF] manager → executor (작업 2/3): 추천 음식점 목록 작성
   ✅ 보고: 어떤 지역의 음식점을 찾으시나요? 또한 매움이나 순함 중 어떤 것을 원하시는지도 알려주세요.

🤝 [HANDOFF] manager → executor (작업 3/3): 1인당 가격 조사
   ✅ 보고: 총 가격과 인원 수를 알려주시면 1인당 가격을 계산하겠습니다.

📋 [최종 종합]
 강남에서 매운 점심을 2명이서 즐길 수 있는 음식점으로 다음과 같은 두 곳을 추천합니다:

1. **매운갈비찜집** - 1인당 가격: 18,000원
2. **불막창** - 1인당 가격: 22,000원

**추천 음식점 및 1인당 가격 요약:**
- 매운갈비찜집: 총 36,000원 (2명 기준)
- 불막창: 총 44,000원 (2명 기준)

이 중에서 원하시는 음식점을 선택하시면 좋겠습니다. 매운 점심을 즐기세요!


## STEP 5 — ⭐ 미니 하네스: 공유 규칙 + 검증 게이트 + 피드백 재시도

여러 에이전트가 돌면 **드리프트**(서로 다른 가정)와 **오류 전파**가 생깁니다. 하네스가 이를 붙잡습니다.

- **① 맥락 주입**: `AGENTS_MD` 규칙을 planner·executor·manager **모두**에게.
- **② 검증 게이트**: 환각·예산·지역 누락·형식 계약을 실제로 검사하고, **어떻게 고칠지도 함께** 알려줍니다.
- **③ 피드백 루프**: 실패한 문제 목록을 **다음 시도의 규칙에 이어붙여** planner가 스스로 교정.

관찰 포인트: `AGENTS_MD` v0는 일부러 **최소한**만 담았습니다. 첫 시도가 실패하면 검증 게이트가 "메뉴 1개 가격·나눠먹기·앵커 형식" 같은 규칙을 **피드백으로 가르치고**, 다음 시도가 그걸 반영합니다 — 하네스가 팀을 **점진적으로 정렬**시키는 모습.

⌨️ **터미널 Codex:** `codex "실패 시 어떻게 고쳐야 하는지까지 담은 validate 게이트를 만들고, 피드백을 다음 시도 planner의 규칙에 이어붙여 3회 재시도 안에 수렴시켜줘"`


In [63]:
import re

# ① AGENTS.md — v0 초기 버전 (일부러 최소한만 — 나머지는 검증 게이트가 가르친다)
AGENTS_MD = """\
# AGENTS.md (팀 공유 규칙 — v0)

## Do
- 사용자 목표에 나온 **모든 지역**에서 실제 데이터의 식당을 하나씩 반드시 고른다.
- 한국어로 답한다.
- 예산 상한을 지킨다.

## Don\'t
- 지어낸 식당 이름을 쓰지 않는다 (환각 금지).
- 조건 맞는 식당이 데이터에 있는데 \"없다\"고 답하지 않는다.
- 여러 옵션을 답에 병기하지 않는다.

## When stuck
- 정보가 부족하면 도구를 한 번 더 호출한다.
"""

# ② 검증 게이트 — 문제와 함께 '고치는 방법(feedback-as-teaching)'까지 담는다
KNOWN_NAMES = {r["name"] for area in RESTAURANTS.values() for r in area}
CITY_RESTAURANTS = {city: {r["name"] for r in items} for city, items in RESTAURANTS.items()}

def _final_per_person(answer):
    """앵커 라인 '최종 ... 1인당 ... 총합 ... N원' 을 관대하게 매칭."""
    for pat in (
        r"최종[^\n]{0,20}1인당[^\n]{0,20}총합[^\d\n]{0,15}([\d,]{2,})\s*원",
        r"최종[^\n]{0,15}1인당[^\d\n]{0,20}([\d,]{2,})\s*원",
    ):
        m = re.search(pat, answer)
        if m:
            return int(m.group(1).replace(",", ""))
    return None

def validate(answer, goal, budget_per_person=None):
    p = []
    if not any("\uac00" <= ch <= "\ud7a3" for ch in answer):
        p.append("한국어 아님")
    # 환각: 답에 나오는 식당형 명사가 DB에 있는지
    cand = re.findall(r"[가-힣]{2,12}(?:집|국밥|파스타|막창|찜|탕|치킨|피자|버거|덮밥)", answer)
    fake = sorted({c for c in cand if c not in KNOWN_NAMES})
    if fake:
        p.append(f"DB에 없는 식당 언급: {fake} — 반드시 실제 데이터의 이름만 쓸 것.")
    # 목표 지역별로 실제 DB 식당 이름을 하나 이상 인용
    for city in ("강남", "홍대"):
        if city in goal:
            names = CITY_RESTAURANTS.get(city, set())
            if not any(n in answer for n in names):
                p.append(
                    f"'{city}'의 실제 식당 미인용 — 반드시 하나 고를 것. "
                    f"선택지: {sorted(names)}"
                )
    # 앵커 라인 계약 검사 — 없으면 정확한 형식을 알려준다
    per = _final_per_person(answer)
    if per is None:
        p.append(
            "출력 형식 규칙 위반 — 답의 **마지막 줄**에 정확히 다음 한 줄을 포함할 것: "
            "`최종 1인당 총합: N원` (예: `최종 1인당 총합: 8,250원`). "
            "이 앵커가 있어야 자동 검증이 가능함."
        )
    elif budget_per_person is not None and per > budget_per_person:
        # Pricing 룰(나눠먹기)을 피드백으로 가르친다
        p.append(
            f"최종 1인당 총합 {per:,}원 > 예산 {budget_per_person:,}원 — "
            "⚠️ RESTAURANTS의 `price`는 **메뉴 1개 가격**이지 1인당이 아니다. "
            "N명이 함께 오면 그 메뉴를 나눠먹어 **1인당 = (선택한 메뉴 가격 합계) ÷ N**. "
            "예: 4명이 18,000원 + 15,000원을 시키면 1인당 = 33,000 ÷ 4 = 8,250원. "
            "`budget_check(price=합계, people=N)` 도구를 호출해 재계산할 것."
        )
    return (len(p) == 0, p)


# ③ 피드백 재시도 루프 — 실패 문제를 다음 시도의 규칙에 이어붙인다
def run_with_harness(goal, budget_per_person=None, max_retries=3):
    print("🛡️ 하네스 시작 — v0 규칙 주입 + 검증 게이트 + 피드백 재시도")
    print("   (초기 규칙은 최소한만. 실패하면 검증 게이트가 무엇을 고칠지 가르친다)\n")
    feedback = []
    final = ""
    for attempt in range(1, max_retries + 1):
        print(f"━━━━━━ 시도 {attempt}/{max_retries} ━━━━━━")
        rules = AGENTS_MD
        if feedback:
            rules += "\n## 이전 시도에서 배운 것 — 반드시 반영\n- " + "\n- ".join(feedback)
        final = manager(goal, rules=rules, verbose=False)
        ok, problems = validate(final, goal, budget_per_person)
        preview = final[:280] + ("…" if len(final) > 280 else "")
        print(f"\n📄 답안 미리보기:\n{preview}\n")
        if ok:
            print(f"🛡️ [VALIDATE] 통과 ✅ (시도 {attempt}에서 수렴)")
            print(f"\n✅ [최종 답]\n{final}")
            return final
        print(f"🛡️ [VALIDATE] 실패 ❌")
        for i, prob in enumerate(problems, 1):
            print(f"   ({i}) {prob}")
        print("   → 위 피드백을 다음 시도의 규칙에 이어붙임 (planner가 스스로 교정)\n")
        feedback = problems
    print("\n⚠️ 검증 미통과 — 사람 검토 필요 (HUMAN REVIEW)")
    print(f"마지막 답:\n{final}")
    return final


# ⭐ 복합 제약 목표 — v0 규칙만으론 한 방에 못 맞춤
COMPLEX_GOAL = (
    "4명이 강남에서 매운 점심 한 곳, 홍대에서 순한 저녁 한 곳으로 하루 코스를 짜줘. "
    "각 식당의 가격과 '1인당 총합'(두 끼 합산)까지 계산해 알려줘."
)
BUDGET_PER_PERSON = 9000  # 원 — 나눠먹기 가정 시 (18000+15000)/4=8250원이면 통과

_ = run_with_harness(COMPLEX_GOAL, budget_per_person=BUDGET_PER_PERSON, max_retries=3)


🛡️ 하네스 시작 — v0 규칙 주입 + 검증 게이트 + 피드백 재시도
   (초기 규칙은 최소한만. 실패하면 검증 게이트가 무엇을 고칠지 가르친다)

━━━━━━ 시도 1/3 ━━━━━━
🧭 [PLAN] 목표를 3 단계로 분해:
   1. 강남에서 매운 점심 식당 찾기
   2. 홍대에서 순한 저녁 식당 찾기
   3. 1인당 총합 계산하기

📄 답안 미리보기:
강남에서 매운 점심으로 "매운갈비찜집"을 추천합니다. 가격은 18,000원입니다.  
홍대에서 순한 저녁으로 "치즈파스타"를 추천합니다. 가격은 15,000원입니다.

1인당 총합은 다음과 같이 계산됩니다:  
- 점심: 18,000원  
- 저녁: 15,000원  
- 총합: 18,000원 + 15,000원 = 33,000원

따라서, 4명이 하루 코스를 진행할 경우 1인당 총합은 33,000원입니다.

🛡️ [VALIDATE] 실패 ❌
   (1) 출력 형식 규칙 위반 — 답의 **마지막 줄**에 정확히 다음 한 줄을 포함할 것: `최종 1인당 총합: N원` (예: `최종 1인당 총합: 8,250원`). 이 앵커가 있어야 자동 검증이 가능함.
   → 위 피드백을 다음 시도의 규칙에 이어붙임 (planner가 스스로 교정)

━━━━━━ 시도 2/3 ━━━━━━
🧭 [PLAN] 목표를 3 단계로 분해:
   1. 강남에서 매운 점심 식당 선택 및 가격 확인
   2. 홍대에서 순한 저녁 식당 선택 및 가격 확인
   3. 식당 가격을 합산하여 1인당 총합 계산

📄 답안 미리보기:
강남에서 매운 점심으로 추천하는 식당은 "매운갈비찜집"이며, 1인당 가격은 18,000원입니다.

홍대에서 순한 저녁으로는 "치즈파스타"를 추천합니다. 이 식당의 1인당 가격은 15,000원입니다.

따라서 4명이 각 식당에서 식사할 경우, 

- 강남에서 매운 점심: 18,000원 
- 홍대에서 순한 저녁: 15,000원 

총합은 18,000원 + 15,000원 = 33,000원 입니다. 

1인당 총합은 33,00

## STEP 6 — 공유 상태(state): 노드마다 '추가'하며 흐른다

강의의 공유 상태 — 각 단계가 상태 딕셔너리에 결과를 **추가**하며 흐르게 만들어, 전체 이력을 한눈에 봅니다 (append, no clobber).

In [64]:
import json as _json

def manager_with_state(goal):
    state = {"goal": goal, "steps": []}
    for i, sub in enumerate(plan(goal), 1):
        result = executor(sub, rules=AGENTS_MD, verbose=False)
        state["steps"].append({"n": i, "subtask": sub, "result": str(result)[:80]})
    state["response"] = state["steps"][-1]["result"] if state["steps"] else ""
    return state

st = manager_with_state("강남에서 매운 맛집을 찾고 2명 예산이 괜찮은지 확인해줘")
print(_json.dumps(st, ensure_ascii=False, indent=2))
# 관찰: 어느 단계가 무엇을 남겼는지 전부 보인다 — 덮어쓰면 이 추적이 불가능해진다

🧭 [PLAN] 목표를 3 단계로 분해:
   1. 강남 지역의 매운 맛집 리스트 찾기
   2. 각 맛집의 메뉴와 가격 확인하기
   3. 2명 예산에 맞는지 비교하기
{
  "goal": "강남에서 매운 맛집을 찾고 2명 예산이 괜찮은지 확인해줘",
  "steps": [
    {
      "n": 1,
      "subtask": "강남 지역의 매운 맛집 리스트 찾기",
      "result": "강남 지역의 매운 맛집이 없습니다."
    },
    {
      "n": 2,
      "subtask": "각 맛집의 메뉴와 가격 확인하기",
      "result": "데이터에서 해당 지역의 식당 정보가 없습니다. 다른 요청을 해주시면 도와드리겠습니다."
    },
    {
      "n": 3,
      "subtask": "2명 예산에 맞는지 비교하기",
      "result": "예산이 얼마인지 말씀해 주시면 그에 맞춰서 비교해 드리겠습니다."
    }
  ],
  "response": "예산이 얼마인지 말씀해 주시면 그에 맞춰서 비교해 드리겠습니다."
}


## STEP 7 — 병렬 작업자: 독립 하위작업은 동시에

강의의 병렬 패턴(map→reduce) — 독립적인 하위작업 2개를 스레드로 동시에 돌려 시간을 비교합니다.

In [65]:
import time
from concurrent.futures import ThreadPoolExecutor

subtasks = ["강남의 매운 맛집을 검색해줘", "홍대의 순한 맛집을 검색해줘"]   # 서로 독립

t0 = time.time()
seq = [executor(st_, verbose=False) for st_ in subtasks]
t_seq = time.time() - t0

t0 = time.time()
with ThreadPoolExecutor() as pool:
    par = list(pool.map(lambda st_: executor(st_, verbose=False), subtasks))
t_par = time.time() - t0

print(f"순차: {t_seq:.1f}s / 병렬: {t_par:.1f}s")
# 관찰: 병렬은 '가장 느린 작업'만큼만 기다린다 — 단, 서로 의존성이 없을 때만

순차: 3.0s / 병렬: 3.2s


## STEP 8 — ⭐ 내 팀 MVP에 적용

위 구조(**도구 → Planner → Executor → Manager → 하네스**)는 **그대로** 당신 팀 MVP로 옮길 수 있습니다.
바꿀 것은 **두 가지뿐**: ① 도구(`TOOLS`/`TOOL_IMPL`)를 우리 서비스 기능으로, ② `AGENTS_MD` 규칙을 우리 팀 기준으로.

⌨️ **터미널 Codex:** `codex "우리 MVP의 핵심 기능 2개를 function tool로 정의하고, 우리 팀 규칙(AGENTS.md)을 넣은 Planner+Executor+검증 게이트로 연결해줘"`

In [66]:
# 팀별 작성 ① — 우리 MVP가 가진 '도구' 2개 (예: 일정앱→add_event, 가계부→add_expense)
MY_TOOLS_TODO = [
  # {"name": "...", "desc": "...", "args": {"...": "..."}},
  # {"name": "...", "desc": "...", "args": {"...": "..."}},
]

# 팀별 작성 ② — 우리 팀의 공유 규칙(미니 AGENTS.md) — 검증 게이트의 기준이 됩니다
MY_RULES_TODO = """\
## Do
- (예) 답에는 반드시 ___ 를 포함한다.
## Don't
- (예) ___ 는 하지 않는다.
## When stuck
- (예) 정보가 부족하면 ___ 한다.
"""

print("우리 MVP 도구 후보:", MY_TOOLS_TODO or "⬜ 채워주세요")
print("우리 팀 규칙:", "✅ 작성됨" if "예)" not in MY_RULES_TODO else "⬜ 채워주세요")
print("→ TOOLS/TOOL_IMPL을 우리 도구로, AGENTS_MD를 MY_RULES_TODO로 바꾸고 run_with_harness 를 다시 돌리면")
print("  '내 팀 MVP 멀티에이전트 + 하네스' 완성")

우리 MVP 도구 후보: ⬜ 채워주세요
우리 팀 규칙: ⬜ 채워주세요
→ TOOLS/TOOL_IMPL을 우리 도구로, AGENTS_MD를 MY_RULES_TODO로 바꾸고 run_with_harness 를 다시 돌리면
  '내 팀 MVP 멀티에이전트 + 하네스' 완성


## (도전) 세 번째 전문가 — Reviewer 추가

관리자-작업자 패턴은 작업자를 늘리기 쉽습니다. `executor`처럼 **reviewer(검토자)** 를 하나 더 만들어,
manager가 최종 답을 reviewer에게 한 번 더 핸드오프해 **품질을 비평·교정**하게 해보세요(= 코드 속 리플렉션).

⌨️ **터미널 Codex:** `codex "manager 종합 직후 최종 답을 검토·교정하는 reviewer 에이전트를 추가해줘"`

## 정리

- **멀티에이전트 = 분할 + 패턴(순차·병렬·관리자) + 통신(핸드오프).** 7차시의 한 에이전트가 **계획자+실행자 팀**이 됐습니다.
- **하네스가 그 협업을 떠받칩니다**: 공유 규칙(`AGENTS_MD`)이 팀을 **정렬**시키고, `validate()` 게이트가 **오류 전파를 막는 가드레일**이 됩니다.
- 에이전트가 여럿일수록 — **검증이 없으면 한 오류가 팀 전체를 오염**시킵니다. 그래서 멀티에이전트엔 하네스가 **필수**입니다.
- 다음 차시(13): 도구·메모리·RAG·멀티·하네스 조각을 **하나의 동작 시스템**으로 통합 → 저녁 팀 프로젝트(Day2)로 바로 적용하세요.